# 01C — ETL Nguon UCI Dropout -> DataMart Chuan Hoa

**Dataset:** UCI Student Dropout (Realinho et al., 2022)
**Input:** `data/raw/uci_dropout/data.csv`
**Output:** `data/processed/uci_dropout_datamart.csv`
**Vai tro OOD:** Tap **TEST DOC LAP 1** — Blind Predict.

---

### So do cot nguon -> DataMart

| Cot DataMart | Cot nguon | Ghi chu |
|---|---|---|
| `admission_score_norm` | `Admission grade` | `/200.0` |
| `priority_flag` | `Educational special needs` | 0/1 |
| `credits_registered` | `enrolled` ky 2 | `x3` (ECTS) |
| `credit_completion_ratio` | `enrolled`, `approved` ky 2 | `approved/enrolled` |
| `gpa_current_semester` | `grade` ky 2 | Piecewise (qua mon=10->GPA 2.0) |
| `cpa_cumulative_score` | grade ky 1+2 | avg -> Piecewise |
| `tuition_debt_flag` | `Tuition fees up to date` | `1-value` |
| `lms_*` | Khong co | **NaN** — OOD Masking |


In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.4f}'.format)

RAW_DIR       = 'data/raw/'       if os.path.basename(os.getcwd()) != 'notebooks' else '../data/raw/'
PROCESSED_DIR = 'data/processed/' if os.path.basename(os.getcwd()) != 'notebooks' else '../data/processed/'
os.makedirs(PROCESSED_DIR, exist_ok=True)
print(f"RAW_DIR: {os.path.abspath(RAW_DIR)}")
print(f"PROCESSED_DIR: {os.path.abspath(PROCESSED_DIR)}")


## 1. Load & Kiem Tra Du Lieu Tho

In [ ]:
df = pd.read_csv(RAW_DIR + 'uci_dropout/data.csv', sep=';')

print(f"Shape: {df.shape}")
print()
print("=== Phan phoi Target ===")
print(df['Target'].value_counts())
print()
print("=== Thong ke diem ky 1 & ky 2 (thang 0-20) ===")
print(df[['Curricular units 1st sem (grade)', 'Curricular units 2nd sem (grade)']].describe())
print()
print("=== Admission grade (thang 0-200) ===")
print(df['Admission grade'].describe())
print()
print("=== Tuition fees up to date ===")
print(df['Tuition fees up to date'].value_counts())
print()
nan_sum = df.isnull().sum()
print("=== NaN ===")
print(nan_sum[nan_sum > 0] if nan_sum.any() else "Khong co NaN")


## 2. Ham Piecewise GPA Alignment

**Van de:** UCI dung thang 0-20, nguong qua mon = 10.
Anh xa tuyen tinh don gian cho nguong qua mon o GPA=2.0 (may trung voi OULAD GPA=1.6).
Piecewise dam bao nguong qua mon luon = **GPA 2.0** nhat quan voi OULAD.


In [ ]:
def map_uci_gpa(g):
    """Piecewise: UCI 0-20 -> GPA 0-4, qua mon (10) -> 2.0"""
    if pd.isna(g): return 0.0
    if g < 10:
        return (g / 10.0) * 2.0
    return 2.0 + ((g - 10.0) / 10.0) * 2.0

test_grades = [0, 5, 10, 12, 14, 17, 20]
print("Grade -> GPA (Piecewise):")
for g in test_grades:
    marker = " <- Nguong qua mon" if g == 10 else ""
    print(f"  grade={g:2d} -> GPA={map_uci_gpa(g):.3f}{marker}")


## 3. Anh Xa -> 22 Cot DataMart

In [ ]:
out = pd.DataFrame()

out['student_code']        = 'UD_' + df.index.astype(str)
out['academic_year_order'] = 1

# Admission: co thuc tren thang 0-200
out['admission_score_norm'] = (df['Admission grade'] / 200.0).clip(0, 1)

out['priority_flag'] = df['Educational special needs']

enrolled_2 = df['Curricular units 2nd sem (enrolled)']
approved_2 = df['Curricular units 2nd sem (approved)']
out['credits_registered']      = enrolled_2 * 3
out['credit_completion_ratio'] = np.where(enrolled_2 > 0, approved_2 / enrolled_2, 0.0)
out['cumulative_credit_debt']  = (enrolled_2 - approved_2).clip(lower=0) * 3

g1 = df['Curricular units 1st sem (grade)']
g2 = df['Curricular units 2nd sem (grade)']
gpa_curr = g2.apply(map_uci_gpa)
gpa_prev = g1.apply(map_uci_gpa)
gpa_avg  = ((g1 + g2) / 2.0).apply(map_uci_gpa)

out['gpa_current_semester'] = gpa_curr
out['cpa_cumulative_score'] = gpa_avg
out['cpa_growth_delta']     = gpa_curr - gpa_prev

out['failed_courses_count'] = (enrolled_2 - approved_2).clip(lower=0)

# training_point_norm: khong co du lieu -> NaN (xu ly o Modeling)
out['training_point_norm'] = np.nan

out['tuition_debt_flag'] = 1 - df['Tuition fees up to date']

# LMS - OOD NaN MASKING: Tuyet doi khong dung median cua OULAD
out['lms_total_clicks'] = np.nan
out['lms_active_days']  = np.nan
out['lms_late_submits'] = np.nan
out['lms_avg_quiz']     = np.nan

out['previous_academic_warnings'] = 0
out['risk_score_continuous'] = 0.0

target_map = {'Graduate': 0, 'Enrolled': 1, 'Dropout': 3}
out['target_risk_level'] = df['Target'].map(target_map).fillna(2).astype(int)
out['target_risk_name']  = out['target_risk_level'].map(
    {0: 'Low', 1: 'Medium', 2: 'High', 3: 'Very High'})
out['dataset_source'] = 'UCI_Dropout'
out = out.reset_index(drop=True)

print(f"Shape sau ETL: {out.shape}")


## 4. Kiem Tra Chat Luong

In [ ]:
print("=== Phan phoi nhan rui ro ===")
print(out['target_risk_name'].value_counts())
print()

print("=== GPA sau Piecewise ===")
print(out['gpa_current_semester'].describe())
pct_pass = (out['gpa_current_semester'] >= 2.0).mean() * 100
print(f"Ti le GPA >= 2.0: {pct_pass:.1f}%")
print()

print("=== Kiem tra LMS = 100% NaN (bat buoc cho OOD) ===")
for col in ['lms_total_clicks', 'lms_active_days', 'lms_late_submits', 'lms_avg_quiz']:
    pct = out[col].isna().mean() * 100
    status = "OK" if pct == 100 else "FAILED"
    print(f"  {col}: {pct:.1f}% NaN [{status}]")
print()

print("=== admission_score_norm ===")
print(out['admission_score_norm'].describe())
print(f"Phuong sai: {out['admission_score_norm'].var():.4f}  <- phai > 0")


## 5. Luu File

In [ ]:
out_path = PROCESSED_DIR + 'uci_dropout_datamart.csv'
out.to_csv(out_path, index=False)
print(f"Da luu: {out_path}")
print(f"Shape : {out.shape}")
